 # Mortgage Decision-Support Model Evaluation



 **DN6MD001 - Artificial Intelligence and Machine Learning**



 This notebook provides evidence of model evaluation, threshold investigation,

 decision-rule testing and applicant validation.



 The mortgage prototype predicts the likelihood of repayment difficulty

 within 12 months. Model output is used for decision support only and

 does not automatically approve or decline a mortgage.

 ## 1. Load mortgage dataset

In [7]:
# from pathlib import Path

# print(Path.cwd())


In [8]:
from src.data import load_data
from src.model import BASELINE_FEATURES

df = load_data()

print("Dataset shape:", df.shape)
print("Model features:")
for feature in BASELINE_FEATURES:
    print("-", feature)



Dataset shape: (800, 19)
Model features:
- annual_income_gbp
- property_value_gbp
- deposit_gbp
- requested_loan_gbp
- loan_to_value_pct
- loan_to_income_ratio
- monthly_debt_commitments_gbp
- debt_to_income_ratio
- credit_score
- employment_years
- savings_gbp
- missed_payments_24m
- credit_utilisation_pct
- previous_default_flag
- term_years
- interest_rate_pct


 ### Interpretation

 The cleaned dataset contains 800 synthetic mortgage applications. Missing values were addressed during preprocessing and

 `loan_to_income_ratio` was introduced as an engineered affordability feature.


 The historical target `repayment_difficulty_12m` is used during training and evaluation only. It is not available when assessing a new applicant.

 ## 2. Examine the target distribution

In [9]:
target_counts = df["repayment_difficulty_12m"].value_counts().sort_index()

print()
print(
    f"No repayment difficulty: "
    f"{target_counts[0]} ({target_counts[0] / len(df):.1%})"
)

print(
    f"Repayment difficulty: "
    f"{target_counts[1]} ({target_counts[1] / len(df):.1%})"
)




No repayment difficulty: 625 (78.1%)
Repayment difficulty: 175 (21.9%)


In [10]:
from src.decisions import advisory_decision

example_application = {
    "annual_income_gbp": 52000,
    "property_value_gbp": 240000,
    "deposit_gbp": 15000,
    "requested_loan_gbp": 225000,
    "loan_to_value_pct": 93.75,
    "loan_to_income_ratio": 225000 / 52000,
    "monthly_debt_commitments_gbp": 157,
    "debt_to_income_ratio": 0.362,
    "credit_score": 702,
    "employment_years": 5,
    "savings_gbp": 4500,
    "missed_payments_24m": 1,
    "credit_utilisation_pct": 37.99,
    "previous_default_flag": 0,
    "term_years": 20,
    "interest_rate_pct": 4.8,
}


tests = [
    (0.00, "LOW RISK"),
    (0.30, "LOW RISK"),
    (0.301, "MEDIUM RISK"),
    (0.60, "MEDIUM RISK"),
    (0.601, "HIGH RISK"),
    (1.00, "HIGH RISK"),
]

for score, expected in tests:
    decision = advisory_decision(
        score,
        example_application
    )

    actual = decision["risk_level"]

    print(
        f"Score={score:.1%} "
        f"Expected={expected} "
        f"Actual={actual} "
        f"PASS={actual == expected}"
    )



Score=0.0% Expected=LOW RISK Actual=LOW RISK PASS=True
Score=30.0% Expected=LOW RISK Actual=LOW RISK PASS=True
Score=30.1% Expected=MEDIUM RISK Actual=MEDIUM RISK PASS=True
Score=60.0% Expected=MEDIUM RISK Actual=MEDIUM RISK PASS=True
Score=60.1% Expected=HIGH RISK Actual=HIGH RISK PASS=True
Score=100.0% Expected=HIGH RISK Actual=HIGH RISK PASS=True


 ### Interpretation

 Boundary testing demonstrates that the advisory decison rules behave correctly at and immediately either side of the 30% and 60% boundaries.

 Risk Outcomes
 - 0-30%: Low Risk

 - Above 30% to 60%: Medium Risk

 - Above 60%: High Risk

 ## 4. Input-validation tests

In [11]:
from src.decisions import validate_application_fields

invalid_tests = {
    "Invalid credit utilisation": {
        **example_application,
        "credit_utilisation_pct": 120,
    },

    "Invalid previous default": {
        **example_application,
        "previous_default_flag": 2,
    },

    "Negative missed payments": {
        **example_application,
        "missed_payments_24m": -1,
    },
}

for test_name, application in invalid_tests.items():
    try:
        validate_application_fields(
            application,
            BASELINE_FEATURES
        )

        print(test_name, "- FAIL: invalid input accepted")

    except ValueError as error:
        print(test_name, "- PASS")
        print(" ", error)



Invalid credit utilisation - PASS
  Credit utilisatipon must be between 0 and 100
Invalid previous default - PASS
  Previous default flag must be true(1) or false(0)
Negative missed payments - PASS
  Missing payments must be a whole number '0' or higher.


 ### Interpretation



 Invalid mortgage-application values are rejected before they are given to the trained model. This prevents obviously invalid input from potentially misleading decision-support output.

 ## 5. Example applicant assessment

In [12]:
from src.applicant_example import score_example

score = score_example(example_application)

decision = advisory_decision(
    score,
    example_application
)

print(f"Model score: {score:.1%}")
print(f"Risk level: {decision['risk_level']}")
print(f"Recommendation: {decision['recommendation']}")

print("\nAdvisory reasoning:")

for reason in decision["reasoning"]:
    print("-", reason)



Model score: 14.9%
Risk level: LOW RISK
Recommendation: Recommended proceed with standard mortgage assessment.

Advisory reasoning:
- Applicant has a total of 1, recorded in the previous 24 months.


 ### Interpretation

 The model score and advisory decison rules serve different purposes.

 The Logistic Regression model produces the class-1 score. The decision layer converts that score into a user-facing risk category and combines it with documented rule-based reasoning.

 The output remains advisory and requires human review.